In [1]:
import pandas as pd
from pathlib import Path
import yaml
import sys

In [2]:
# ========== Opening config file ==========

path_yaml = Path("../src/config.yaml")
try:

    with open(path_yaml, "r") as file:
        config = yaml.safe_load(file)

except FileNotFoundError:
    print("Config file not found")
    sys.exit(1)

In [3]:
# ========== Creating path variables and opening dataset ==========

path_raw = Path(config["paths"]["raw"])
path_processed = Path(config["paths"]["processed"])
useless_columns = config["features"]["useless_columns"]
raw_standardization = config["data_standardization"]
extra_services_columns = config["features"]["extra_services"]

df = pd.read_csv(path_raw)

pd.set_option('future.no_silent_downcasting', True) # Boilerplate so i dont get stupid logs

In [4]:
# ========== Removing Churn clients and useless columns ==========


df = df[df["Churn Value"] == 0]

df = df.drop(columns=useless_columns)

In [5]:
# ========== Standardizing the data based on yaml file ==========

mapping = config["mapping"]
real_mapping = {}

for k, v in mapping.items():
    real_mapping[v] = k


df = df.rename(columns=real_mapping)

map_to_apply = {}
for std_key, raw_values in config["data_standardization"].items():
    if isinstance(raw_values, list):
        for val in raw_values:
            map_to_apply[val] = std_key
    else:
        map_to_apply[raw_values] = std_key

df = df.replace(map_to_apply)

In [6]:
# ========== Creating usefull columns and formating the entrys to lower case ==========

df["is_fiber"] = (df["internet_service"] == "fiber_optic").astype(int)
df = df.drop(columns=["internet_service"])

for col in config["features"]["nominal_categoricals"]:
    df[col] = df[col].astype(str).str.lower().str.strip()

df = df.replace({"yes":1, "no":0})

In [7]:
# ========== Dumping the dataframe into a csv file ==========

df.set_index("customer_id", inplace=True)

df.to_csv(path_processed, index=True)

In [8]:
df

,gender,senior_citizen,partner,dependents,tenure_months,phone_service,multiple_lines,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,contract_type,paperless_billing,payment_method,cltv,is_fiber
customer_id,,,,,,,,,,,,,,,,,,
7590-VHVEG,female,0,1,0,1,0,0,0,1,0,0,0,0,month_to_month,1,electronic_check,3964,0
5575-GNVDE,male,0,0,0,34,1,0,1,0,1,0,0,0,one_year,0,mailed_check,3441,0
7795-CFOCW,male,0,0,0,45,0,0,1,0,1,1,0,0,one_year,0,bank_transfer,4307,0
1452-KIOVK,male,0,0,1,22,1,1,0,1,0,0,1,0,month_to_month,1,credit_card,4459,1
6713-OKOMC,female,0,0,0,10,0,0,1,0,0,0,0,0,month_to_month,0,mailed_check,2013,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2569-WGERO,female,0,0,0,72,1,0,0,0,0,0,0,0,two_year,1,bank_transfer,5306,0
6840-RESVB,male,0,1,1,24,1,1,1,0,1,1,1,1,one_year,1,mailed_check,2140,0
2234-XADUH,female,0,1,1,72,1,1,0,1,1,0,1,1,one_year,1,credit_card,5560,1
